### Realtime Human Pose Detection

In [ ]:
import cv2
import mediapipe as mp

In [4]:
## Initialse pose estimator
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence = 0.5)

In [ ]:
## Pose estimation for video 
cap = cv2.VideoCapture('test_video.mp4')
while cap.isOpened() :
    ## Read Frame ##
    _, frame = cap.read() ### to get each frame one by one 
    try :
        #### resizing the frame for potrait video ###
        frame = cv2.resize(frame, (350, 600)) ##[resolution change]
        ### converting to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        ### process the frames for pose detection ~
        pose_results = pose.process(frame_rgb)

        #print(pose_results.pose_landmarks)
        ### Draw skeleton on the frame 
        mp_drawing.draw_landmarks(frame, pose_results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        ### displaying the frame 
        cv2.imshow('Output', frame)
    except :
        break
    if cv2.waitKey(1) == ord('q') : 
        break
cap.release()
cv2.destroyAllWindows()

In [ ]:
### To get Landmark for specific point 
print(pose_results.pose_landmarks[0]) ##0-33

In [8]:
### Realtime Pose detection
## Pose estimation for video 
cap = cv2.VideoCapture(0)
while cap.isOpened() :
    ## Read Frame ##
    _, frame = cap.read() ### to get each frame one by one 
    try :
        #### resizing the frame for potrait video ###
        # frame = cv2.resize(frame, (350, 600)) ##[resolution change]
        ### converting to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        ### process the frames for pose detection 
        pose_results = pose.process(frame_rgb)

        #print(pose_results.pose_landmarks)
        ### Draw skeleton on the frame 
        mp_drawing.draw_landmarks(frame, pose_results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        ### displaying the frame 
        cv2.imshow('Output', frame)
    except :
        break
    if cv2.waitKey(1) == ord('q') : 
        break
cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import mediapipe as mp
import numpy as np

# Initialize MediaPipe Pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)
mp_drawing = mp.solutions.drawing_utils

def calculate_angle(a, b, c):
    """
    Calculates the angle between three points (e.g., a joint).
    
    Args:
        a (tuple): Coordinates of the first point (e.g., shoulder).
        b (tuple): Coordinates of the second point (e.g., elbow). This is the vertex.
        c (tuple): Coordinates of the third point (e.g., wrist).
        
    Returns:
        float: The calculated angle in degrees.
    """
    a = np.array(a)  # First point
    b = np.array(b)  # Mid point (vertex)
    c = np.array(c)  # End point
    
    # Calculate vectors
    ba = a - b
    bc = c - b
    
    # Calculate the dot product and magnitudes
    dot_product = np.dot(ba, bc)
    magnitude_ba = np.linalg.norm(ba)
    magnitude_bc = np.linalg.norm(bc)
    
    # Calculate the cosine of the angle
    cosine_angle = dot_product / (magnitude_ba * magnitude_bc)
    
    # Calculate the angle in radians and then convert to degrees
    angle = np.arccos(cosine_angle)
    
    return np.degrees(angle)

# --- Main Video Capture Loop ---
cap = cv2.VideoCapture(0)  # 0 for webcam

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("Ignoring empty camera frame.")
        continue

    # Convert the BGR image to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Process the frame for pose detection
    pose_results = pose.process(frame_rgb)
    
    # Draw the pose annotation on the frame
    if pose_results.pose_landmarks:
        mp_drawing.draw_landmarks(
            frame, pose_results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        
        # --- Angle Calculation & Form Checking ---
        try:
            # Get landmarks
            landmarks = pose_results.pose_landmarks.landmark
            
            # --- Left Arm ---
            left_shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,
                             landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
            left_elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x,
                          landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
            left_wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x,
                          landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]
            
            # Calculate angle
            left_elbow_angle = calculate_angle(left_shoulder, left_elbow, left_wrist)
            
            # --- Display the Angle ---
            # Create a visual text box on the frame
            cv2.putText(frame, f"Left Elbow: {int(left_elbow_angle)}", 
                        tuple(np.multiply(left_elbow, [640, 480]).astype(int)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)

            # --- Form Quality Check (Example for a bicep curl) ---
            form_status = "INCORRECT FORM"
            box_color = (0, 0, 255) # Red for incorrect
            
            # Check if the arm is relatively straight (start/end of curl)
            if left_elbow_angle > 160:
                form_status = "CORRECT FORM (Down)"
                box_color = (0, 255, 0) # Green for correct

            elif left_elbow_angle < 30:
                form_status = "CORRECT FORM (Up)"
                box_color = (0, 255, 0) # Green for correct

            # Display Form Status
            cv2.rectangle(frame, (10, 10), (350, 60), box_color, -1)
            cv2.putText(frame, f"STATUS: {form_status}", (20, 40), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
            
        except Exception as e:
            # print(f"Error: {e}") # Uncomment to debug
            pass # Pass if landmarks are not visible


    cv2.imshow('MediaPipe Pose Form Checker', frame)

    if cv2.waitKey(5) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt


mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

def calculate_angle(a, b, c):
    """Calculates the angle between three points."""
    a = np.array(a)  # First
    b = np.array(b)  # Mid
    c = np.array(c)  # End
    
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    
    if angle > 180.0:
        angle = 360 - angle
        
    return angle

def moving_average(data, window_size):
    """Calculates the moving average of a list."""
    if not data:
        return []
    return np.convolve(data, np.ones(window_size), 'valid') / window_size

# --- Video Analysis ---
video_path = 'test_video.mp4'
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print(f"Error: Could not open video file {video_path}")
    exit()

form_history = []  # List to store form status (1 for correct, 0 for incorrect)
frame_count = 0

print("Analyzing video... Please wait.")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break # End of video

    # Convert BGR to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Process for pose
    pose_results = pose.process(frame_rgb)
    
    is_correct = 0 
    
    if pose_results.pose_landmarks:
        try:
            landmarks = pose_results.pose_landmarks.landmark
            

            left_shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,
                             landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
            left_elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x,
                          landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
            left_wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x,
                          landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]
            
            # Calculate angle
            angle = calculate_angle(left_shoulder, left_elbow, left_wrist)
            
            # Form check (bicep curl example)
            if angle > 160 or angle < 30:
                is_correct = 1 # Form is correct
            else:
                is_correct = 0 # Form is incorrect
        
        except:
            is_correct = 0 # Landmark not visible, count as incorrect

    form_history.append(is_correct)
    frame_count += 1

    print(f"Processing frame: {frame_count}", end='\r')


cap.release()
print("\nAnalysis complete.")

# --- Graph Generation ---
if not form_history:
    print("No data was collected to generate a graph.")
else:
    # Calculate overall percentage
    total_frames = len(form_history)
    correct_frames = sum(form_history)
    correct_percentage = (correct_frames / total_frames) * 100

    print(f"Overall Form Correctness: {correct_percentage:.2f}%")

    # Generate the graph
    plt.style.use('seaborn-v0_8-darkgrid') # Use a nice style
    fig, ax = plt.subplots(figsize=(12, 6))

    # Calculate a moving average for a smoother line
    window_size = 50 # Adjust window size for more or less smoothing
    if len(form_history) > window_size:
        avg_form = moving_average(form_history, window_size)
        ax.plot(avg_form, label=f'Form Correctness (Moving Avg. over {window_size} frames)', color='cyan')
    else:
        # Plot raw data if video is too short for moving average
        ax.plot(form_history, label='Form Correctness (Raw)', color='cyan', alpha=0.5)

    # Fill area under the curve
    ax.fill_between(range(len(avg_form)), avg_form, color="cyan", alpha=0.3)

    # Set titles and labels
    ax.set_title('Form Quality Analysis Over Time', fontsize=16)
    ax.set_xlabel('Frame Sequence', fontsize=12)
    ax.set_ylabel('Correctness Score', fontsize=12)
    ax.set_ylim(0, 1.1)
    ax.set_yticks([0, 0.5, 1.0])
    ax.set_yticklabels(['Incorrect', 'Okay', 'Correct'])

    # Add a text box with the final score
    score_text = f'Overall Correctness: {correct_percentage:.2f}%'
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
    ax.text(0.05, 0.95, score_text, transform=ax.transAxes, fontsize=14,
            verticalalignment='top', bbox=props)
            
    plt.legend()
    plt.tight_layout()
    plt.show()